# Python–MATLAB comparison through line 167

This notebook evaluates the translated Python code in the same order as `KLJCmainsv22.m`. Each calculated variable is displayed with its value, shape, and data type. The final cells save the Python variables and compare them with variables exported from MATLAB at line 167.

In [11]:
from pathlib import Path
import sys

import numpy as np
from scipy.io import loadmat, savemat

current_directory = Path.cwd().resolve()
possible_project_directories = [
    current_directory,
    current_directory / "pyKlyC",
    current_directory.parent,
]
project_directory = next(
    (path for path in possible_project_directories if (path / "src").exists()),
    None,
)
if project_directory is None:
    raise FileNotFoundError("Could not find the pyKlyC project directory.")

notebook_directory = project_directory / "Notebooks"
field_map_directory = notebook_directory / "field_maps"
field_map_directory.mkdir(exist_ok=True)

sys.path.insert(0, str(project_directory / "src"))

from constants import c, esign, eta0, etom, etom_signed, mu01, mu0
from model.beam import BeamParameters
from model.cavity import CavityParameters
from model.settings import NumericalSettings
from model.tube import TubeParameters
from solver.beam_state import BeamState
from solver.cavity_fields import CavityFields
from solver.grid import GridParameters
from solver.reference import ReferenceParameters
from solver.space_charge import (
    calculate_plasma_frequency,
    calculate_prefill_space_charge,
)

np.set_printoptions(precision=12, threshold=np.inf, linewidth=160)

In [12]:
def show_variable(name, value):
    array = np.asarray(value)
    if array.ndim == 0:
        print(f"{name}: {array.item()!r} ({array.dtype})")
    else:
        print(f"{name}: shape={array.shape}, dtype={array.dtype}")
        print(array)
    print()


def show_object(obj):
    for name in obj.__dataclass_fields__:
        show_variable(name, getattr(obj, name))

## Test inputs

This is a small deterministic test case. Replace the values in this cell with the values used by your MATLAB case when needed. Python indices in `Y0` start at zero.

In [13]:
factor = 1.0
factorregion = 10.0
fillfactor = 1.0
DCSPfactor = 1.0

Nt = 16
t0 = 2.0 * np.pi * np.arange(Nt) / Nt

numerical = NumericalSettings(
    f=12.0,
    Nc=1,
    Ns=3,
    Nz=20,
    Nlimit=20,
    Ntimes=2,
    Rlimit=1.0e-6,
    iterfactor=0.5,
)

beam = BeamParameters(
    Nbeam=1,
    rm2=1.0,
    rm1=0.0,
    rc=3.0,
    xb=0.0,
    yb=0.0,
    xd=0.0,
    yd=0.0,
    Nt=Nt,
    t0=t0,
    Ut=50.0 * (1.0 + 0.02 * np.sin(t0)),
    Izt=-10.0 * (1.0 + 0.10 * np.cos(t0)),
    tn1=t0.copy(),
    Y0=np.arange(Nt),
)

cavities = CavityParameters(
    type=np.array([1]),
    nh=np.array([1]),
    f0=np.array([12.0]),
    Qext=np.array([1000.0]),
    Qin=np.array([10000.0]),
    zshift=np.array([10.0]),
    Pin=np.array([0.0]),
    gaman=np.array([0.0 + 0.0j]),
    cavityname="synthetic_cavity",
    kai=np.zeros((1, 1), dtype=complex),
)

tube = TubeParameters(
    z0=0.0,
    L=20.0,
    rcv=np.array([[0.0, 20.0], [1.0, 1.0]]),
)

## Synthetic cavity field map

The same text file can be loaded by MATLAB. Its first column is the position relative to `zshift`; its second column is the longitudinal field.

In [14]:
field_position = np.linspace(-2.0, 2.0, 81)
field_value = np.exp(-(field_position / 0.75) ** 2)
field_map = np.column_stack((field_position, field_value))
field_map_path = field_map_directory / "synthetic_cavity.txt"
np.savetxt(field_map_path, field_map)
show_variable("field_map", field_map)

field_map: shape=(81, 2), dtype=float64
[[-2.000000000000e+00  8.159878350721e-04]
 [-1.950000000000e+00  1.159229173905e-03]
 [-1.900000000000e+00  1.632279495219e-03]
 [-1.850000000000e+00  2.278029544742e-03]
 [-1.800000000000e+00  3.151111598444e-03]
 [-1.750000000000e+00  4.320239474094e-03]
 [-1.700000000000e+00  5.870721275738e-03]
 [-1.650000000000e+00  7.907054051593e-03]
 [-1.600000000000e+00  1.055546956620e-02]
 [-1.550000000000e+00  1.396625646626e-02]
 [-1.500000000000e+00  1.831563888873e-02]
 [-1.450000000000e+00  2.380694872273e-02]
 [-1.400000000000e+00  3.067079302639e-02]
 [-1.350000000000e+00  3.916389509899e-02]
 [-1.300000000000e+00  4.956628350491e-02]
 [-1.250000000000e+00  6.217652402212e-02]
 [-1.200000000000e+00  7.730474044330e-02]
 [-1.150000000000e+00  9.526325532781e-02]
 [-1.100000000000e+00  1.163548028087e-01]
 [-1.050000000000e+00  1.408584209210e-01]
 [-1.000000000000e+00  1.690133154061e-01]
 [-9.500000000000e-01  2.010011912257e-01]
 [-9.000000000

## Constants and reference variables

In [15]:
for name, value in {
    "c": c,
    "etom_signed": etom_signed,
    "etom": etom,
    "esign": esign,
    "eta0": eta0,
    "mu01": mu01,
    "mu0": mu0,
}.items():
    show_variable(name, value)

reference = ReferenceParameters.calculate_reference_parameters(beam)
show_object(reference)

c: 299792458.0 (float64)

etom_signed: -175880000000.0 (float64)

etom: 175880000000.0 (float64)

esign: 1.0 (float64)

eta0: 376.73031346177066 (float64)

mu01: 2.40483 (float64)

mu0: shape=(5,), dtype=float64
[ 2.40483  5.52008  8.65373 11.79153 14.93092]

Iz0: -10.0 (float64)

I0: 10.0 (float64)

U0: 50.0 (float64)

s: 0.412684137102372 (float64)

vetoc: shape=(16,), dtype=float64
[0.412684137102 0.41405189378  0.415205713881 0.415973778243 0.416242939876 0.415973778243 0.415205713881 0.41405189378  0.412684137102 0.411309016293
 0.410137416574 0.40935157114  0.409075043918 0.40935157114  0.410137416574 0.411309016293]

vn1: shape=(16,), dtype=float64
[1.             0.996696653975 0.993926921777 0.992091710313 0.991450178651 0.992091710313 0.993926921777 0.996696653975 1.             1.003343279032
 1.006209432317 1.008141085067 1.008822569936 1.008141085067 1.006209432317 1.003343279032]

gama: 1.0978464459293553 (float64)



## Frequency and longitudinal-grid variables

In [16]:
grid = GridParameters.calculate(
    numerical,
    cavities,
    tube,
    beam,
    reference,
    factorregion,
)
show_object(grid)

Nesz_matlab = grid.Nesz.copy()
Nesz_matlab[:, 0] += 1
show_variable("Nesz_matlab", Nesz_matlab)

w: 75398223686.15503 (float64)

w0: shape=(1,), dtype=float64
[7.539822368616e+10]

k0: shape=(1,), dtype=float64
[0.251501402634]

Lsb: shape=(1,), dtype=float64
[3.14159265359]

betae: 0.609428325498766 (float64)

dz: 0.5263157894736842 (float64)

zi: shape=(39,), dtype=float64
[ 0.              0.526315789474  1.052631578947  1.578947368421  2.105263157895  2.631578947368  3.157894736842  3.684210526316  4.210526315789
  4.736842105263  5.263157894737  5.789473684211  6.315789473684  6.842105263158  7.368421052632  7.894736842105  8.421052631579  8.947368421053
  9.473684210526 10.             10.526315789474 11.052631578947 11.578947368421 12.105263157895 12.631578947368 13.157894736842 13.684210526316
 14.210526315789 14.736842105263 15.263157894737 15.789473684211 16.315789473684 16.842105263158 17.368421052632 17.894736842105 18.421052631579
 18.947368421053 19.473684210526 20.            ]

z: shape=(39,), dtype=float64
[ 0.              0.320751750263  0.641503500525  0.962255

`Nesz` stores zero-based Python slice bounds, with an exclusive stop. `Nesz_matlab` contains the equivalent one-based, inclusive MATLAB indices.

## Cavity-field variables

In [17]:
cavity_fields = CavityFields.calculate(
    numerical,
    cavities,
    grid,
    reference,
    field_map_directory,
)
show_object(cavity_fields)

ez: shape=(39, 1), dtype=float64
[[0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.011991590366]
 [0.13956875681 ]
 [0.611108967383]
 [1.            ]
 [0.611108967383]
 [0.13956875681 ]
 [0.011991590366]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]]

C3: shape=(39, 1), dtype=float64
[[0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.            ]
 [0.          

## Initial beam-state variables

In [18]:
state = BeamState.calculate(numerical, beam, grid, reference)
show_object(state)

Iz: shape=(39, 3), dtype=complex128
[[ 1.387778780781e-17+1.000000000000e-01j -6.346672767815e-17+6.687413163582e-18j -6.245004513517e-16-6.383782391595e-16j]
 [ 1.387778780781e-17+1.000000000000e-01j -6.346672767815e-17+6.687413163582e-18j -6.245004513517e-16-6.383782391595e-16j]
 [ 1.387778780781e-17+1.000000000000e-01j -6.346672767815e-17+6.687413163582e-18j -6.245004513517e-16-6.383782391595e-16j]
 [ 1.387778780781e-17+1.000000000000e-01j -6.346672767815e-17+6.687413163582e-18j -6.245004513517e-16-6.383782391595e-16j]
 [ 1.387778780781e-17+1.000000000000e-01j -6.346672767815e-17+6.687413163582e-18j -6.245004513517e-16-6.383782391595e-16j]
 [ 1.387778780781e-17+1.000000000000e-01j -6.346672767815e-17+6.687413163582e-18j -6.245004513517e-16-6.383782391595e-16j]
 [ 1.387778780781e-17+1.000000000000e-01j -6.346672767815e-17+6.687413163582e-18j -6.245004513517e-16-6.383782391595e-16j]
 [ 1.387778780781e-17+1.000000000000e-01j -6.346672767815e-17+6.687413163582e-18j -6.245004513517e-16-6

## Plasma frequency and upstream pre-fill variables

In [19]:
wp = calculate_plasma_frequency(beam, reference, factor)
show_variable("wp", wp)

zp = grid.z[0] - grid.dt * np.arange(1, grid.dN + 1)
Nfill = len(zp)
gamazp = np.ones((Nfill, numerical.Ns + 1))
gamazp[:, 1:] = state.gamaz[0]

QQfill = np.zeros((Nfill, numerical.Ns + 1), dtype=complex)
QQfill[:, 0] = state.QQDC[0]
harmonics = np.arange(1, numerical.Ns + 1)
QQfill[:, 1:] = state.QQ[0] * np.exp(
    -1.0j * harmonics * (grid.z[0] - zp[:, np.newaxis])
)

zp_matlab = np.repeat(zp[:, np.newaxis], numerical.Ns + 1, axis=1)

show_variable("zp", zp)
show_variable("Nfill", Nfill)
show_variable("gamazp", gamazp)
show_variable("QQfill", QQfill)

Esz00 = calculate_prefill_space_charge(
    beam,
    grid,
    reference,
    state,
    wp,
    fillfactor,
    DCSPfactor,
)
show_variable("Esz00", Esz00)

wp: 22606831004.258602 (float64)

zp: shape=(23,), dtype=float64
[-0.320751750263 -0.641503500525 -0.962255250788 -1.28300700105  -1.603758751313 -1.924510501575 -2.245262251838 -2.5660140021   -2.886765752363
 -3.207517502625 -3.528269252888 -3.84902100315  -4.169772753413 -4.490524503675 -4.811276253938 -5.1320280042   -5.452779754463 -5.773531504725
 -6.094283254988 -6.41503500525  -6.735786755513 -7.056538505775 -7.377290256038]

Nfill: 23 (int64)

gamazp: shape=(23, 4), dtype=float64
[[1.             1.097846445929 1.097846445929 1.097846445929]
 [1.             1.097846445929 1.097846445929 1.097846445929]
 [1.             1.097846445929 1.097846445929 1.097846445929]
 [1.             1.097846445929 1.097846445929 1.097846445929]
 [1.             1.097846445929 1.097846445929 1.097846445929]
 [1.             1.097846445929 1.097846445929 1.097846445929]
 [1.             1.097846445929 1.097846445929 1.097846445929]
 [1.             1.097846445929 1.097846445929 1.097846445929]
 [

## Save the Python checkpoint

In [10]:
python_variables = {
    "Iz0": reference.Iz0,
    "I0": reference.I0,
    "U0": reference.U0,
    "s": reference.s,
    "vetoc": reference.vetoc,
    "vn1": reference.vn1,
    "gama": reference.gama,
    "w": grid.w,
    "w0": grid.w0,
    "k0": grid.k0,
    "Lsb": grid.Lsb,
    "betae": grid.betae,
    "dz": grid.dz,
    "zi": grid.zi,
    "z": grid.z,
    "dt": grid.dt,
    "rcvs": grid.rcvs,
    "dN": grid.dN,
    "Nesz": Nesz_matlab,
    "ez": cavity_fields.ez,
    "C3": cavity_fields.C3,
    "vpz": cavity_fields.vpz,
    "alfaz": cavity_fields.alfaz,
    "Iz": state.Iz,
    "QQ": state.QQ,
    "IzDC": state.IzDC,
    "QQDC": state.QQDC,
    "wp": wp,
    "effe": state.effe,
    "vez": state.vez,
    "gamaz": state.gamaz,
    "Esz00": Esz00,
    "zp": zp_matlab,
    "Nfill": Nfill,
    "gamazp": gamazp,
    "QQfill": QQfill,
}

python_file = notebook_directory / "python_checkpoint_167.mat"
savemat(python_file, python_variables, oned_as="column")
print(f"Saved {python_file}")

Saved C:\Users\lsito\Desktop\PyKlyC\pyKlyC\Notebooks\python_checkpoint_167.mat


## Export the MATLAB checkpoint

Set a breakpoint at line 167 of `KLJCmainsv22.m`. Once execution reaches it, save the corresponding workspace variables as `matlab_checkpoint_167.mat` inside `pyKlyC/Notebooks`:

```matlab
save('matlab_checkpoint_167.mat', ...
    'Iz0','I0','U0','s','vetoc','vn1','gama', ...
    'w','w0','k0','Lsb','betae','dz','zi','z','dt','rcvs','dN','Nesz', ...
    'ez','C3','vpz','alfaz','Iz','QQ','IzDC','QQDC','wp', ...
    'effe','vez','gamaz','Esz00','zp','Nfill','gamazp','QQfill');
```

## Compare Python with MATLAB

In [ ]:
matlab_file = notebook_directory / "matlab_checkpoint_167.mat"

if not matlab_file.exists():
    print(f"Place the MATLAB checkpoint at {matlab_file}")
else:
    matlab_variables = loadmat(matlab_file)

    for name, python_value in python_variables.items():
        if name not in matlab_variables:
            print(f"{name}: missing from MATLAB file")
            continue

        python_array = np.squeeze(np.asarray(python_value))
        matlab_array = np.squeeze(np.asarray(matlab_variables[name]))

        if python_array.shape != matlab_array.shape:
            print(
                f"{name}: shape mismatch, "
                f"Python {python_array.shape}, MATLAB {matlab_array.shape}"
            )
            continue

        difference = np.abs(python_array - matlab_array)
        maximum_error = float(np.max(difference)) if difference.size else 0.0
        matches = np.allclose(
            python_array,
            matlab_array,
            rtol=1.0e-10,
            atol=1.0e-12,
            equal_nan=True,
        )
        print(f"{name}: match={matches}, max abs error={maximum_error:.6e}")